# Objective Quest DATAQUEST 5.0 — V5 Boundary-1 Breakthrough

**Goal:** stop making incremental 4-class GBDT variants and directly attack the bottleneck identified from V1–V4: separating **class 1 (Sangat Puas)** from **class >1**.

This notebook is intentionally **rule-safe** and self-contained:

- uses only competition train/test data;
- **does not use pretrained tabular foundation weights**;
- uses a strong all-raw-categorical CatBoost ordinal anchor;
- probes three boundary-1 specialists: all-cat CatBoost, **TabM + piecewise-linear numerical embeddings**, and **RealMLP-TD**;
- validates on **three different outer folds** before any full 5-fold run;
- blends a boundary specialist into the ordinal anchor using calibration data only;
- full CV runs only after a qualitative jump;
- submission is created only when honest unique-ID cross-fitted QWK reaches the current target **0.61541**.

The notebook is designed for a Kaggle **T4×2** accelerator. CatBoost can use both GPUs; TabM and RealMLP run on separate CUDA devices when available.

## 0. Install modern tabular packages

Kaggle usually does not preinstall the research packages used here. The cell installs them only when missing. If Internet is disabled in the Kaggle notebook, enable it for package installation. No pretrained model weights are downloaded.

In [1]:
import sys, subprocess, importlib.util

def ensure_package(import_name, pip_name):
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {pip_name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

ensure_package('tabm', 'tabm')
ensure_package('rtdl_num_embeddings', 'rtdl_num_embeddings')
ensure_package('pytabkit', 'pytabkit==1.7.3')
print('Research packages available.')

Installing tabm ...
Installing pytabkit==1.7.3 ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 9.3 MB/s eta 0:00:00
Research packages available.


## 1. Imports and reproducibility

In [2]:
import os, gc, json, math, random, copy, time, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.special import expit, logit

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, roc_auc_score, log_loss, mean_squared_error

from catboost import CatBoostRegressor, CatBoostClassifier

import torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from tabm import TabM
from rtdl_num_embeddings import PiecewiseLinearEmbeddings, compute_bins

try:
    from pytabkit import RealMLP_TD_Classifier
    REALMLP_AVAILABLE = True
except Exception as e:
    print('RealMLP import unavailable:', repr(e))
    REALMLP_AVAILABLE = False

warnings.filterwarnings('ignore')
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

torch 2.10.0+cu128 | CUDA True | GPUs 2
0 Tesla T4
1 Tesla T4


## 2. Configuration

In [3]:
CFG = {
    'VERSION': 'v5_boundary_breakthrough',
    'TARGET': 'life_satisfaction',
    'ID_COL': 'id',
    'HOUSEHOLD_COL': 'household_id',
    'SEED': 42,
    'N_FOLDS': 5,
    'PROBE_FOLDS': [0, 2, 3],
    'INNER_CAL_FOLDS': 5,

    # Raw-code feature typing.
    'CAT_MAX_UNIQUE': 120,

    # CatBoost anchor / specialist.
    'CAT_PROBE_ITER': 1800,
    'CAT_FULL_ITER': 4500,
    'CAT_EARLY_STOP': 160,
    'CAT_DEPTH': 8,
    'CAT_LR': 0.035,
    'CAT_DEVICES': '0:1',

    # TabM specialist.
    'TABM_PROBE_EPOCHS': 45,
    'TABM_FULL_EPOCHS': 70,
    'TABM_PATIENCE': 7,
    'TABM_BATCH': 1024,
    'TABM_K': 16,
    'TABM_BLOCKS': 2,
    'TABM_D_BLOCK': 256,
    'TABM_NUM_BINS': 32,
    'TABM_NUM_EMBED': 12,
    'TABM_LR': 0.002,
    'TABM_WEIGHT_DECAY': 3e-4,

    # RealMLP specialist.
    'REALMLP_PROBE_EPOCHS': 128,
    'REALMLP_FULL_EPOCHS': 256,
    'REALMLP_BATCH': 512,
    'REALMLP_PROBE_ENS': 2,
    'REALMLP_FULL_ENS': 4,

    # Calibration / specialist injection.
    'ALPHA_GRID_MAX': 3.0,
    'ALPHA_GRID_N': 31,
    'B1_WEIGHT_RANDOM': 120,

    # Research targets derived from V1–V4 oracle analysis.
    'CURRENT_TOP1_PUBLIC': 0.61541,
    'V2_PUBLIC_LB': 0.48193,
    'V2_CROSSFIT': 0.481367,
    'TARGET_B1_AUC': 0.893,

    # Multi-probe promotion: no more +0.001 promotions.
    'PROMOTE_MIN_B1_AUC': 0.850,
    'PROMOTE_MIN_BLEND_QWK': 0.530,
    'PROMOTE_MIN_WINS': 3,
    'PROMOTE_MIN_QWK_GAIN': 0.015,

    # Submission stays locked until honest CV meets the requested threshold.
    'SUBMISSION_QWK_GATE': 0.61541,

    'EXPECTED_TRAIN_ROWS': 95609,
    'EXPECTED_TEST_ROWS': 23902,
    'EXPECTED_UNIQUE_TEST_IDS': 23666,
}
OUT = Path('/kaggle/working/oqdq5_v5')
OUT.mkdir(parents=True, exist_ok=True)
print(CFG)

{'VERSION': 'v5_boundary_breakthrough', 'TARGET': 'life_satisfaction', 'ID_COL': 'id', 'HOUSEHOLD_COL': 'household_id', 'SEED': 42, 'N_FOLDS': 5, 'PROBE_FOLDS': [0, 2, 3], 'INNER_CAL_FOLDS': 5, 'CAT_MAX_UNIQUE': 120, 'CAT_PROBE_ITER': 1800, 'CAT_FULL_ITER': 4500, 'CAT_EARLY_STOP': 160, 'CAT_DEPTH': 8, 'CAT_LR': 0.035, 'CAT_DEVICES': '0:1', 'TABM_PROBE_EPOCHS': 45, 'TABM_FULL_EPOCHS': 70, 'TABM_PATIENCE': 7, 'TABM_BATCH': 1024, 'TABM_K': 16, 'TABM_BLOCKS': 2, 'TABM_D_BLOCK': 256, 'TABM_NUM_BINS': 32, 'TABM_NUM_EMBED': 12, 'TABM_LR': 0.002, 'TABM_WEIGHT_DECAY': 0.0003, 'REALMLP_PROBE_EPOCHS': 128, 'REALMLP_FULL_EPOCHS': 256, 'REALMLP_BATCH': 512, 'REALMLP_PROBE_ENS': 2, 'REALMLP_FULL_ENS': 4, 'ALPHA_GRID_MAX': 3.0, 'ALPHA_GRID_N': 31, 'B1_WEIGHT_RANDOM': 120, 'CURRENT_TOP1_PUBLIC': 0.61541, 'V2_PUBLIC_LB': 0.48193, 'V2_CROSSFIT': 0.481367, 'TARGET_B1_AUC': 0.893, 'PROMOTE_MIN_B1_AUC': 0.85, 'PROMOTE_MIN_BLEND_QWK': 0.53, 'PROMOTE_MIN_WINS': 3, 'PROMOTE_MIN_QWK_GAIN': 0.015, 'SUBMISSION_Q

## 3. Load data and fix the evaluator contract

The Kaggle evaluator expects one row per **unique `test_final.id`**, not one row per raw test row. Duplicate-ID rows were previously verified to be exact duplicates. V5 deduplicates by ID at the beginning so CV and evaluation match the leaderboard unit directly.

In [4]:

def discover_data_dir():
    roots=[Path('/kaggle/input/competitions/oq-dq-5'), Path('/kaggle/input')]
    for root in roots:
        if not root.exists(): continue
        if (root/'train_final.csv').exists() and (root/'test_final.csv').exists(): return root
        for p in root.rglob('train_final.csv'):
            if (p.parent/'test_final.csv').exists(): return p.parent
    for root in [Path('/mnt/data/_raw_oq'), Path('/mnt/data/oqdq5'), Path('.')]:
        if (root/'train_final.csv').exists() and (root/'test_final.csv').exists(): return root
    raise FileNotFoundError('train_final.csv/test_final.csv not found')

DATA_DIR=discover_data_dir()
train_raw=pd.read_csv(DATA_DIR/'train_final.csv')
test_raw=pd.read_csv(DATA_DIR/'test_final.csv')
dictionary=pd.read_csv(DATA_DIR/'data_dictionary.csv') if (DATA_DIR/'data_dictionary.csv').exists() else pd.DataFrame()

assert len(train_raw)==CFG['EXPECTED_TRAIN_ROWS']
assert len(test_raw)==CFG['EXPECTED_TEST_ROWS']
assert test_raw[CFG['ID_COL']].nunique()==CFG['EXPECTED_UNIQUE_TEST_IDS']
assert train_raw.groupby(CFG['ID_COL'])[CFG['TARGET']].nunique().max()==1

train_counts=train_raw[CFG['ID_COL']].value_counts()
test_counts=test_raw[CFG['ID_COL']].value_counts()
train=train_raw.drop_duplicates(CFG['ID_COL'],keep='first').copy()
test=test_raw.drop_duplicates(CFG['ID_COL'],keep='first').copy()
train['_id_repeat_count']=train[CFG['ID_COL']].map(train_counts).astype(np.int8)
test['_id_repeat_count']=test[CFG['ID_COL']].map(test_counts).astype(np.int8)

FUNCTIONAL=[
    'difficulty_walking_steps','difficulty_walking_short','difficulty_walking_medium','difficulty_climbing_steps',
    'difficulty_using_hands','difficulty_raising_arm','difficulty_self_care','difficulty_communicating',
    'difficulty_remembering','difficulty_errands_alone','difficulty_social_activities']
CHRONIC=['hypertension_ever','high_cholesterol_ever','heart_disease_ever','heart_attack_ever','angina_ever','stroke_ever',
         'asthma_ever','diabetes_ever','prediabetes_ever','cancer_ever','lung_disease_ever','arthritis_ever','dementia_ever','chronic_fatigue_ever']
COST=['trouble_paying_medical_bills','delayed_care_due_to_cost','skipped_care_due_to_cost','skipped_medication_to_save_money',
      'took_less_medication_to_save_money','delayed_prescription_to_save_money','skipped_prescription_due_to_cost',
      'delayed_therapy_due_to_cost','skipped_therapy_due_to_cost']
INCOME=['has_wage_income','has_investment_income','has_social_security_income','has_disability_income','has_public_assistance_income','has_retirement_income','has_other_income']
INSURANCE=['has_health_insurance','has_private_insurance','has_senior_gov_insurance','has_lowincome_gov_insurance','has_child_gov_insurance','has_military_insurance','has_state_health_plan','has_other_gov_insurance']
SPECIAL_NUMERIC_CODES={
    'weight_pounds':{996,997,999}, 'height_inches':{96,97,98,99}, 'age':{97,99},
    'hours_worked_per_week':{990}, 'days_missed_work':{997,998,999},
}

def _count_yes(df,cols):
    cc=[c for c in cols if c in df.columns]
    return (df[cc]==1).sum(axis=1).astype(np.int16) if cc else pd.Series(0,index=df.index,dtype=np.int16)

def _reverse_ordinal(s,max_valid):
    x=pd.to_numeric(s,errors='coerce'); out=pd.Series(np.nan,index=s.index,dtype=float)
    m=x.between(1,max_valid); out.loc[m]=(max_valid+1)-x.loc[m]
    return out

def add_static_features(df):
    out=df.copy()
    base_cols=[c for c in train_raw.columns if c not in {CFG['TARGET'],CFG['ID_COL'],CFG['HOUSEHOLD_COL']} and c in out.columns]
    out['_missing_count']=out[base_cols].isna().sum(axis=1).astype(np.int16)
    out['functional_difficulty_count']=_count_yes(out,FUNCTIONAL)
    out['chronic_condition_count']=_count_yes(out,CHRONIC)
    out['cost_barrier_count']=_count_yes(out,COST)
    out['income_source_count']=_count_yes(out,INCOME)
    out['insurance_count']=_count_yes(out,INSURANCE)
    food=[]
    for c in ['worried_food_would_run_out','food_did_not_last','could_not_afford_balanced_meals']:
        if c in out: food.append(_reverse_ordinal(out[c],3).fillna(0))
    out['food_insecurity_count']=(sum(food) if food else 0).astype(np.float32) if food else np.float32(0)
    if 'received_food_assistance' in out:
        out['food_insecurity_count']=pd.Series(out['food_insecurity_count'],index=out.index).astype(np.float32)+(out['received_food_assistance']==1).astype(np.float32)
    mental=[]
    for c in ['depression_frequency','anxiety_frequency']:
        if c in out: mental.append(_reverse_ordinal(out[c],5).fillna(0))
    out['mental_frequency_burden']=(sum(mental) if mental else pd.Series(0,index=out.index)).astype(np.float32)
    hh=[]
    for c in ['children_in_household','adults_in_household']:
        if c in out: hh.append(pd.to_numeric(out[c],errors='coerce').fillna(0))
    out['household_size_proxy']=(sum(hh) if hh else pd.Series(0,index=out.index)).astype(np.float32)
    if 'age' in out:
        age=pd.to_numeric(out['age'],errors='coerce').where(pd.to_numeric(out['age'],errors='coerce').between(18,85))
        out['age_band']=pd.cut(age,bins=[17,24,34,44,54,64,74,85],labels=False,include_lowest=True).astype(float)
    for c,codes in SPECIAL_NUMERIC_CODES.items():
        if c in out:
            out[c+'__special_code']=pd.to_numeric(out[c],errors='coerce').isin(codes).astype(np.int8)
    return out

train=add_static_features(train)
test=add_static_features(test)

assert train[CFG['ID_COL']].is_unique
assert test[CFG['ID_COL']].is_unique
assert len(test)==CFG['EXPECTED_UNIQUE_TEST_IDS']
print('DATA_DIR:',DATA_DIR)
print('raw train/test:',train_raw.shape,test_raw.shape)
print('unique-ID engineered train/test:',train.shape,test.shape)
print(train[CFG['TARGET']].value_counts(normalize=True).sort_index())


DATA_DIR: /kaggle/input/competitions/oq-dq-5
raw train/test: (95609, 179) (23902, 178)
unique-ID engineered train/test: (94663, 195) (23666, 194)
life_satisfaction
1    0.449510
2    0.504030
3    0.037089
4    0.009370
Name: proportion, dtype: float64


## 4. CV: same unit as the leaderboard

In [5]:
TARGET,ID_COL,HH_COL=CFG['TARGET'],CFG['ID_COL'],CFG['HOUSEHOLD_COL']
y=train[TARGET].astype(int).to_numpy()
y_b1=(y>1).astype(np.int8)

# Stratify jointly by target and survey year. IDs are unique after deduplication.
year=train.get('survey_year',pd.Series('NA',index=train.index)).astype(str)
strata=(train[TARGET].astype(str)+'_'+year).to_numpy()
skf=StratifiedKFold(n_splits=CFG['N_FOLDS'],shuffle=True,random_state=CFG['SEED'])
folds=list(skf.split(train,strata))

for f,(tr,va) in enumerate(folds):
    print(f'fold {f}: train={len(tr):,} val={len(va):,} positive_B1={y_b1[va].mean():.4f}')

def inner_fit_cal_split(outer_train_idx,seed):
    sub=train.iloc[outer_train_idx]
    sub_strata=(sub[TARGET].astype(str)+'_'+sub.get('survey_year',pd.Series('NA',index=sub.index)).astype(str)).to_numpy()
    inner=StratifiedKFold(n_splits=CFG['INNER_CAL_FOLDS'],shuffle=True,random_state=seed)
    fit_rel,cal_rel=next(inner.split(sub,sub_strata))
    return np.asarray(outer_train_idx)[fit_rel],np.asarray(outer_train_idx)[cal_rel]

fold 0: train=75,730 val=18,933 positive_B1=0.5505
fold 1: train=75,730 val=18,933 positive_B1=0.5505
fold 2: train=75,730 val=18,933 positive_B1=0.5505
fold 3: train=75,731 val=18,932 positive_B1=0.5505
fold 4: train=75,731 val=18,932 positive_B1=0.5505


## 5. QWK, thresholding, and specialist injection

In [6]:
def qwk(y_true,y_pred):
    return cohen_kappa_score(np.asarray(y_true,int),np.asarray(y_pred,int),weights='quadratic')

def apply_thresholds(score,thresholds):
    return np.digitize(np.asarray(score,float),np.sort(np.asarray(thresholds,float)))+1

def optimize_thresholds(y_true,score,x0=(1.55,2.03,2.52)):
    y_true=np.asarray(y_true,int); score=np.asarray(score,float)
    def obj(z):
        z=np.sort(z)
        if np.min(np.diff(z))<0.025: return 10.0
        return -qwk(y_true,apply_thresholds(score,z))
    starts=[x0,(1.45,1.95,2.40),(1.60,2.10,2.65)]
    best=None
    for s in starts:
        r=minimize(obj,np.array(s,float),method='Nelder-Mead',options={'maxiter':550,'xatol':1e-5,'fatol':1e-6})
        th=np.sort(r.x); sc=-obj(th)
        if best is None or sc>best[0]: best=(sc,th)
    return best[1]

def auc_b1(y_true,score):
    return roc_auc_score((np.asarray(y_true)>1).astype(int),np.asarray(score,float))

def _safe_logit(p):
    return logit(np.clip(np.asarray(p,float),1e-5,1-1e-5))

def tune_specialist_injection(y_cal,base_cal,p1_cal):
    # Calibration-only standardization. Higher base and higher p1 both mean worse satisfaction.
    b=np.asarray(base_cal,float); l=_safe_logit(p1_cal)
    bm,bs=b.mean(),max(b.std(),1e-8); lm,ls=l.mean(),max(l.std(),1e-8)
    zb=(b-bm)/bs; zl=(l-lm)/ls
    best=None
    for a in np.linspace(0,CFG['ALPHA_GRID_MAX'],CFG['ALPHA_GRID_N']):
        s=zb+a*zl
        th=optimize_thresholds(y_cal,s,x0=(-0.45,0.55,1.55))
        sc=qwk(y_cal,apply_thresholds(s,th))
        if best is None or sc>best['cal_qwk']:
            best={'cal_qwk':float(sc),'alpha':float(a),'thresholds':th,
                  'base_mean':bm,'base_std':bs,'logit_mean':lm,'logit_std':ls}
    return best

def apply_specialist_injection(base,p1,tune):
    zb=(np.asarray(base,float)-tune['base_mean'])/tune['base_std']
    zl=(_safe_logit(p1)-tune['logit_mean'])/tune['logit_std']
    return zb+tune['alpha']*zl

## 6. Feature views

V2/V3 overused dictionary-defined "valid" values. V5 does the opposite: **observed raw codes are the source of truth**.

- Low-cardinality survey-coded variables are categorical tokens.
- True/likely continuous variables stay numeric.
- A curated set of genuinely ordered health/economic variables is represented **twice** for neural models: categorical token + numeric ordinal copy.
- Missing numeric values are median-imputed only for neural models, with explicit missing indicators. Tree models keep native missing values.

In [7]:

EXCLUDE={TARGET,ID_COL,HH_COL}
raw_features=[c for c in train.columns if c not in EXCLUDE]
all_card=pd.concat([train[raw_features],test[raw_features]],ignore_index=True).nunique(dropna=True)

# Numerical geometry is kept only where it is defensible. Survey design IDs remain categorical.
NUMERIC_PRIMARY=[c for c in [
    'income_to_poverty_ratio','survey_weight','hours_worked_per_week','days_missed_work',
    'height_inches','weight_pounds','age','emergency_room_visits','years_in_current_home',
    'children_in_family','children_in_household','adults_in_family','adults_in_household',
    'cancer_types_count','_id_repeat_count','_missing_count','functional_difficulty_count',
    'chronic_condition_count','cost_barrier_count','income_source_count','insurance_count',
    'food_insecurity_count','mental_frequency_burden','household_size_proxy','age_band'
] if c in raw_features]
NUMERIC_PRIMARY += [c for c in raw_features if c.endswith('__special_code')]

ORDINAL_DUAL=[c for c in [
    'general_health_status','depression_frequency','anxiety_frequency','anxiety_level',
    'food_security_3cat','food_security_4cat','income_poverty_category','education_level',
    'difficulty_social_activities','difficulty_remembering','difficulty_errands_alone',
    'difficulty_walking_steps','difficulty_walking_short','difficulty_walking_medium',
    'difficulty_climbing_steps','difficulty_using_hands','difficulty_raising_arm',
    'difficulty_self_care','difficulty_communicating','difficulty_seeing','difficulty_hearing',
    'work_limited_by_health','worry_about_medical_costs','time_since_last_doctor_visit'
] if c in raw_features]

CONSTANT_LIKE={'record_type','adult_sample_flag','imputation_number','household_respondent_flag'}
CAT_FEATURES=[c for c in raw_features if all_card[c] <= CFG['CAT_MAX_UNIQUE'] and c not in CONSTANT_LIKE and c not in NUMERIC_PRIMARY]
NUM_FEATURES=list(dict.fromkeys(NUMERIC_PRIMARY+ORDINAL_DUAL))

print('raw + engineered features:',len(raw_features))
print('categorical token features:',len(CAT_FEATURES),'sum cardinality ~',int(sum(all_card[c]+2 for c in CAT_FEATURES)))
print('numeric features:',len(NUM_FEATURES),'of which dual ordinal:',len(ORDINAL_DUAL))


raw + engineered features: 192
categorical token features: 158 sum cardinality ~ 1300
numeric features: 54 of which dual ordinal: 24


## 7. CatBoost raw-categorical anchor and B1 specialist

In [8]:
def make_catboost_frame(df):
    X=pd.DataFrame(index=df.index)
    for c in raw_features:
        if c in CAT_FEATURES:
            X[c]=df[c].astype('string').fillna('__NA__').astype(str)
        else:
            X[c]=pd.to_numeric(df[c],errors='coerce')
    return X

CB_TRAIN=make_catboost_frame(train)
CB_TEST=make_catboost_frame(test)
CB_CAT_IDX=[CB_TRAIN.columns.get_loc(c) for c in CAT_FEATURES if c in CB_TRAIN.columns]

def fit_cat_anchor(fit_idx,cal_idx,full=False,seed=42):
    it=CFG['CAT_FULL_ITER'] if full else CFG['CAT_PROBE_ITER']
    params=dict(
        loss_function='RMSE',eval_metric='RMSE',iterations=it,learning_rate=CFG['CAT_LR'],
        depth=CFG['CAT_DEPTH'],l2_leaf_reg=5.0,random_seed=seed,verbose=False,
        allow_writing_files=False,task_type='GPU' if torch.cuda.is_available() else 'CPU',
    )
    if torch.cuda.is_available(): params['devices']=CFG['CAT_DEVICES']
    m=CatBoostRegressor(**params)
    m.fit(CB_TRAIN.iloc[fit_idx],y[fit_idx],cat_features=CB_CAT_IDX,
          eval_set=(CB_TRAIN.iloc[cal_idx],y[cal_idx]),early_stopping_rounds=CFG['CAT_EARLY_STOP'])
    return m

def fit_cat_b1(fit_idx,cal_idx,full=False,seed=42):
    it=CFG['CAT_FULL_ITER'] if full else CFG['CAT_PROBE_ITER']
    params=dict(
        loss_function='Logloss',eval_metric='AUC',iterations=it,learning_rate=CFG['CAT_LR'],
        depth=CFG['CAT_DEPTH'],l2_leaf_reg=5.0,random_seed=seed,verbose=False,
        allow_writing_files=False,task_type='GPU' if torch.cuda.is_available() else 'CPU',
    )
    if torch.cuda.is_available(): params['devices']=CFG['CAT_DEVICES']
    m=CatBoostClassifier(**params)
    m.fit(CB_TRAIN.iloc[fit_idx],y_b1[fit_idx],cat_features=CB_CAT_IDX,
          eval_set=(CB_TRAIN.iloc[cal_idx],y_b1[cal_idx]),early_stopping_rounds=CFG['CAT_EARLY_STOP'])
    return m

## 8. TabM + piecewise-linear numerical embeddings

TabM is trained **from scratch**. Categorical variables are raw observed tokens. Numerical variables use robust scaling, explicit missing flags, and quantile-based piecewise-linear embeddings. The loss is computed independently for all TabM ensemble members, then probabilities are averaged only at inference.

In [9]:
class TabMPreprocessor:
    def __init__(self,cat_cols,num_cols):
        self.cat_cols=list(cat_cols); self.num_cols=list(num_cols)
    def fit(self,df):
        self.cat_maps={}; self.cardinalities=[]
        for c in self.cat_cols:
            s=df[c].astype('string').fillna('__NA__').astype(str)
            vals=pd.Index(s.unique())
            mp={v:i+1 for i,v in enumerate(vals)} # 0 = unknown
            self.cat_maps[c]=mp; self.cardinalities.append(len(mp)+1)
        self.num_median={}; self.num_iqr={}; self.num_keep=[]; self.missing_flag_cols=[]
        for c in self.num_cols:
            x=pd.to_numeric(df[c],errors='coerce').astype(float)
            if c in SPECIAL_NUMERIC_CODES: x=x.mask(x.isin(SPECIAL_NUMERIC_CODES[c]))
            med=float(x.median()) if x.notna().any() else 0.0
            q1=float(x.quantile(.25)) if x.notna().any() else 0.0
            q3=float(x.quantile(.75)) if x.notna().any() else 1.0
            iqr=max(q3-q1,1e-3)
            filled=((x.fillna(med)-med)/iqr).clip(-12,12)
            if filled.nunique(dropna=False)>1:
                self.num_keep.append(c); self.num_median[c]=med; self.num_iqr[c]=iqr
            if x.isna().any() and (~x.isna()).any(): self.missing_flag_cols.append(c)
        return self
    def transform(self,df):
        cats=[]
        for c in self.cat_cols:
            s=df[c].astype('string').fillna('__NA__').astype(str)
            mp=self.cat_maps[c]
            cats.append(s.map(mp).fillna(0).astype(np.int64).to_numpy())
        xcat=np.column_stack(cats).astype(np.int64) if cats else np.zeros((len(df),0),np.int64)
        nums=[]
        for c in self.num_keep:
            x=pd.to_numeric(df[c],errors='coerce').astype(float)
            if c in SPECIAL_NUMERIC_CODES: x=x.mask(x.isin(SPECIAL_NUMERIC_CODES[c]))
            arr=((x.fillna(self.num_median[c])-self.num_median[c])/self.num_iqr[c]).clip(-12,12).astype(np.float32).to_numpy()
            nums.append(arr)
        for c in self.missing_flag_cols:
            xmiss=pd.to_numeric(df[c],errors='coerce').astype(float)
            if c in SPECIAL_NUMERIC_CODES: xmiss=xmiss.mask(xmiss.isin(SPECIAL_NUMERIC_CODES[c]))
            nums.append(xmiss.isna().astype(np.float32).to_numpy())
        xnum=np.column_stack(nums).astype(np.float32) if nums else np.zeros((len(df),0),np.float32)
        return xnum,xcat

def _tabm_predict(model,xnum,xcat,device,batch=4096):
    ds=TensorDataset(torch.from_numpy(xnum),torch.from_numpy(xcat))
    dl=DataLoader(ds,batch_size=batch,shuffle=False,num_workers=0,pin_memory=(device.type=='cuda'))
    out=[]; model.eval()
    with torch.no_grad():
        for xn,xc in dl:
            xn=xn.to(device,non_blocking=True); xc=xc.to(device,non_blocking=True)
            logits=model(xn,xc).squeeze(-1) # B,k
            p=torch.sigmoid(logits).mean(dim=1)
            out.append(p.float().cpu().numpy())
    return np.concatenate(out)

def fit_tabm_b1(fit_idx,cal_idx,full=False,seed=42):
    if not torch.cuda.is_available():
        raise RuntimeError('TabM V5 is intended for a Kaggle GPU accelerator.')
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    device=torch.device('cuda:0')
    prep=TabMPreprocessor(CAT_FEATURES,NUM_FEATURES).fit(train.iloc[fit_idx])
    xn_fit,xc_fit=prep.transform(train.iloc[fit_idx]); xn_cal,xc_cal=prep.transform(train.iloc[cal_idx])

    # Piecewise-linear bins are computed on the fit partition only.
    bins=compute_bins(torch.from_numpy(xn_fit),n_bins=min(CFG['TABM_NUM_BINS'],max(4,len(fit_idx)//1000)))
    emb=PiecewiseLinearEmbeddings(bins,CFG['TABM_NUM_EMBED'],activation=False,version='B')
    try:
        model=TabM.make(n_num_features=xn_fit.shape[1],cat_cardinalities=prep.cardinalities,
                        num_embeddings=emb,d_out=1,k=CFG['TABM_K'],
                        n_blocks=CFG['TABM_BLOCKS'],d_block=CFG['TABM_D_BLOCK'])
    except TypeError:
        model=TabM.make(n_num_features=xn_fit.shape[1],cat_cardinalities=prep.cardinalities,
                        num_embeddings=emb,d_out=1)
    model=model.to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG['TABM_LR'],weight_decay=CFG['TABM_WEIGHT_DECAY'])
    scaler=torch.cuda.amp.GradScaler(enabled=True)
    epochs=CFG['TABM_FULL_EPOCHS'] if full else CFG['TABM_PROBE_EPOCHS']

    ds=TensorDataset(torch.from_numpy(xn_fit),torch.from_numpy(xc_fit),torch.from_numpy(y_b1[fit_idx].astype(np.float32)))
    dl=DataLoader(ds,batch_size=CFG['TABM_BATCH'],shuffle=True,num_workers=0,pin_memory=True,drop_last=False)

    best_auc=-1; best_state=None; bad=0
    for epoch in range(epochs):
        model.train(); losses=[]
        for xn,xc,yy in dl:
            xn=xn.to(device,non_blocking=True); xc=xc.to(device,non_blocking=True); yy=yy.to(device,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=True):
                logits=model(xn,xc).squeeze(-1) # B,k
                target=yy[:,None].expand_as(logits)
                loss=F.binary_cross_entropy_with_logits(logits,target,reduction='mean')
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(float(loss.detach().cpu()))
        pcal=_tabm_predict(model,xn_cal,xc_cal,device)
        a=roc_auc_score(y_b1[cal_idx],pcal)
        if a>best_auc+1e-5:
            best_auc=a; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
        if epoch==0 or (epoch+1)%5==0:
            print(f'TabM epoch {epoch+1:02d} loss={np.mean(losses):.5f} cal_auc={a:.6f} best={best_auc:.6f}')
        if bad>=CFG['TABM_PATIENCE']: break
    model.load_state_dict(best_state); model=model.to(device)
    return model,prep,device,best_auc

def predict_tabm_dataframe(model,prep,device,df):
    xn,xc=prep.transform(df)
    return _tabm_predict(model,xn,xc,device)

## 9. RealMLP-TD boundary specialist

RealMLP uses its tuned-default training recipe. Numeric missing values are fold-median-imputed because PyTabKit requires finite numeric inputs; categorical missing values remain explicit tokens. Strongly ordinal features receive an additional numeric copy while the raw code remains categorical.

In [10]:
class RealMLPPreprocessor:
    def __init__(self):
        self.cat_cols=list(CAT_FEATURES)
    def fit(self,df):
        self.num_cols=[c for c in raw_features if c not in self.cat_cols]
        self.medians={}
        for c in self.num_cols:
            x=pd.to_numeric(df[c],errors='coerce').astype(float)
            if c in SPECIAL_NUMERIC_CODES: x=x.mask(x.isin(SPECIAL_NUMERIC_CODES[c]))
            self.medians[c]=float(x.median()) if x.notna().any() else 0.0
        self.ordinal_medians={c:float(pd.to_numeric(df[c],errors='coerce').median()) if pd.to_numeric(df[c],errors='coerce').notna().any() else 0.0 for c in ORDINAL_DUAL}
        return self
    def transform(self,df):
        X=pd.DataFrame(index=np.arange(len(df)))
        for c in self.cat_cols:
            X[c]=df[c].astype('string').fillna('__NA__').astype(str).to_numpy()
        for c in self.num_cols:
            x=pd.to_numeric(df[c],errors='coerce').astype(float)
            if c in SPECIAL_NUMERIC_CODES: x=x.mask(x.isin(SPECIAL_NUMERIC_CODES[c]))
            X[c]=x.fillna(self.medians[c]).astype(np.float32).to_numpy()
            if x.isna().any(): X[c+'__missing']=x.isna().astype(np.float32).to_numpy()
        for c in ORDINAL_DUAL:
            X[c+'__ordnum']=pd.to_numeric(df[c],errors='coerce').fillna(self.ordinal_medians[c]).astype(np.float32).to_numpy()
        return X

def fit_realmlp_b1(fit_idx,cal_idx,full=False,seed=42):
    if not REALMLP_AVAILABLE: raise RuntimeError('RealMLP unavailable')
    prep=RealMLPPreprocessor().fit(train.iloc[fit_idx])
    xf=prep.transform(train.iloc[fit_idx]); xc=prep.transform(train.iloc[cal_idx])
    cat_cols=[c for c in prep.cat_cols if c in xf.columns]
    device='cuda:1' if torch.cuda.is_available() and torch.cuda.device_count()>1 else ('cuda:0' if torch.cuda.is_available() else 'cpu')
    model=RealMLP_TD_Classifier(
        device=device,random_state=seed,n_cv=1,n_refit=0,
        n_epochs=CFG['REALMLP_FULL_EPOCHS'] if full else CFG['REALMLP_PROBE_EPOCHS'],
        batch_size=CFG['REALMLP_BATCH'],
        n_ens=CFG['REALMLP_FULL_ENS'] if full else CFG['REALMLP_PROBE_ENS'],
        val_metric_name='1-auc_ovr',use_ls=False,verbosity=0,
    )
    model.fit(xf,y_b1[fit_idx],xc,y_b1[cal_idx],cat_col_names=cat_cols)
    return model,prep

def predict_realmlp(model,prep,df):
    p=model.predict_proba(prep.transform(df))
    p=np.asarray(p)
    return p[:,1] if p.ndim==2 else p.reshape(-1)

## 10. B1 specialist ensemble

The oracle analysis suggested that QWK ≈0.615 becomes plausible only when the `y>1` separator approaches roughly **0.893 AUC**. The notebook therefore also tests a calibration-only cross-family B1 ensemble. Ensemble weights are selected on the calibration split and then frozen for the untouched outer validation fold.

In [11]:
def search_b1_weights(y_cal,preds_dict,seed):
    names=list(preds_dict)
    P=np.column_stack([_safe_logit(preds_dict[n]) for n in names])
    candidates=[]
    for i in range(len(names)):
        w=np.zeros(len(names)); w[i]=1; candidates.append(w)
    candidates.append(np.ones(len(names))/len(names))
    if len(names)>=2:
        for i in range(len(names)):
            for j in range(i+1,len(names)):
                w=np.zeros(len(names)); w[i]=w[j]=0.5; candidates.append(w)
    rng=np.random.default_rng(seed)
    for _ in range(CFG['B1_WEIGHT_RANDOM']): candidates.append(rng.dirichlet(np.ones(len(names))))
    best=None
    for w in candidates:
        z=P@w
        a=roc_auc_score(y_cal,z)
        if best is None or a>best['auc']:
            best={'auc':float(a),'weights':w.copy(),'names':names.copy()}
    return best

def apply_b1_weights(preds_dict,fit):
    P=np.column_stack([_safe_logit(preds_dict[n]) for n in fit['names']])
    return expit(P@fit['weights'])

## 11. Multi-probe Boundary-1 shootout

In [12]:
probe_records=[]
probe_fold_cache={}
SPECIALISTS=['catboost_b1','tabm_ple_b1'] + (['realmlp_b1'] if REALMLP_AVAILABLE else [])

for fold_id in CFG['PROBE_FOLDS']:
    print('\n'+'='*100+f'\nPROBE FOLD {fold_id}\n'+'='*100)
    outer_tr,outer_va=folds[fold_id]
    fit_idx,cal_idx=inner_fit_cal_split(outer_tr,5000+fold_id)
    ycal=y[cal_idx]; yva=y[outer_va]

    # One ordinal anchor per fold, reused by all specialists.
    t0=time.time()
    anchor=fit_cat_anchor(fit_idx,cal_idx,full=False,seed=5100+fold_id)
    base_cal=anchor.predict(CB_TRAIN.iloc[cal_idx]); base_va=anchor.predict(CB_TRAIN.iloc[outer_va])
    base_th=optimize_thresholds(ycal,base_cal)
    base_q=qwk(yva,apply_thresholds(base_va,base_th)); base_auc=auc_b1(yva,base_va)
    print(f'Anchor QWK={base_q:.6f} B1_AUC={base_auc:.6f} time={time.time()-t0:.1f}s')

    cal_preds={}; val_preds={}

    # CatBoost B1.
    t=time.time(); m=fit_cat_b1(fit_idx,cal_idx,full=False,seed=5200+fold_id)
    cal_preds['catboost_b1']=m.predict_proba(CB_TRAIN.iloc[cal_idx])[:,1]
    val_preds['catboost_b1']=m.predict_proba(CB_TRAIN.iloc[outer_va])[:,1]
    del m; gc.collect()
    print('CatBoost B1 done',time.time()-t)

    # TabM + PLE B1.
    try:
        t=time.time(); m,prep,dev,_=fit_tabm_b1(fit_idx,cal_idx,full=False,seed=5300+fold_id)
        cal_preds['tabm_ple_b1']=predict_tabm_dataframe(m,prep,dev,train.iloc[cal_idx])
        val_preds['tabm_ple_b1']=predict_tabm_dataframe(m,prep,dev,train.iloc[outer_va])
        del m,prep; torch.cuda.empty_cache(); gc.collect()
        print('TabM B1 done',time.time()-t)
    except Exception as e:
        print('TabM candidate failed and will be skipped:',repr(e))

    # RealMLP B1.
    if REALMLP_AVAILABLE:
        try:
            t=time.time(); m,prep=fit_realmlp_b1(fit_idx,cal_idx,full=False,seed=5400+fold_id)
            cal_preds['realmlp_b1']=predict_realmlp(m,prep,train.iloc[cal_idx])
            val_preds['realmlp_b1']=predict_realmlp(m,prep,train.iloc[outer_va])
            del m,prep; torch.cuda.empty_cache(); gc.collect()
            print('RealMLP B1 done',time.time()-t)
        except Exception as e:
            print('RealMLP candidate failed and will be skipped:',repr(e))

    # Individual candidates.
    for name in list(cal_preds):
        pcal,pva=cal_preds[name],val_preds[name]
        tune=tune_specialist_injection(ycal,base_cal,pcal)
        score_va=apply_specialist_injection(base_va,pva,tune)
        pred_va=apply_thresholds(score_va,tune['thresholds'])
        rec={
            'candidate':name,'fold':fold_id,'anchor_qwk':base_q,'blend_qwk':qwk(yva,pred_va),
            'gain_qwk':qwk(yva,pred_va)-base_q,
            'anchor_b1_auc':base_auc,'specialist_b1_auc':roc_auc_score(y_b1[outer_va],pva),
            'blend_b1_auc':auc_b1(yva,score_va),'alpha':tune['alpha'],
            'residual_corr_vs_anchor':float(np.corrcoef(yva-base_va,yva-score_va)[0,1]),
        }
        probe_records.append(rec); print(rec)

    # Cross-family B1 ensemble if at least two specialists survived.
    if len(cal_preds)>=2:
        wf=search_b1_weights(y_b1[cal_idx],cal_preds,5500+fold_id)
        pcal=apply_b1_weights(cal_preds,wf); pva=apply_b1_weights(val_preds,wf)
        tune=tune_specialist_injection(ycal,base_cal,pcal)
        score_va=apply_specialist_injection(base_va,pva,tune)
        pred_va=apply_thresholds(score_va,tune['thresholds'])
        rec={
            'candidate':'b1_cross_family','fold':fold_id,'anchor_qwk':base_q,'blend_qwk':qwk(yva,pred_va),
            'gain_qwk':qwk(yva,pred_va)-base_q,'anchor_b1_auc':base_auc,
            'specialist_b1_auc':roc_auc_score(y_b1[outer_va],pva),'blend_b1_auc':auc_b1(yva,score_va),
            'alpha':tune['alpha'],'residual_corr_vs_anchor':float(np.corrcoef(yva-base_va,yva-score_va)[0,1]),
            'b1_weights':{n:float(w) for n,w in zip(wf['names'],wf['weights'])},
        }
        probe_records.append(rec); print(rec)

    probe_fold_cache[fold_id]={'available_specialists':list(cal_preds)}
    del anchor; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

probe_df=pd.DataFrame(probe_records)
probe_df.to_csv(OUT/'v5_probe_results.csv',index=False)
print('\nProbe results:')
print(probe_df.to_string(index=False))


PROBE FOLD 0
Anchor QWK=0.468415 B1_AUC=0.772964 time=94.9s


Default metric period is 5 because AUC is/are not implemented for GPU


CatBoost B1 done 90.15488171577454
TabM epoch 01 loss=0.59640 cal_auc=0.774448 best=0.774448
TabM epoch 05 loss=0.55439 cal_auc=0.779021 best=0.779021
TabM epoch 10 loss=0.52839 cal_auc=0.777697 best=0.779174
TabM B1 done 43.23204779624939


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=128` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics 

RealMLP B1 done 209.41298913955688
{'candidate': 'catboost_b1', 'fold': 0, 'anchor_qwk': np.float64(0.4684145364922969), 'blend_qwk': np.float64(0.4671913110452679), 'gain_qwk': np.float64(-0.0012232254470290194), 'anchor_b1_auc': np.float64(0.7729644959672786), 'specialist_b1_auc': np.float64(0.7739370935207717), 'blend_b1_auc': np.float64(0.7740446338073426), 'alpha': 0.5, 'residual_corr_vs_anchor': 0.40919741895361184}
{'candidate': 'tabm_ple_b1', 'fold': 0, 'anchor_qwk': np.float64(0.4684145364922969), 'blend_qwk': np.float64(0.4703258352463502), 'gain_qwk': np.float64(0.00191129875405327), 'anchor_b1_auc': np.float64(0.7729644959672786), 'specialist_b1_auc': np.float64(0.7755010217285492), 'blend_b1_auc': np.float64(0.7742312030706264), 'alpha': 0.2, 'residual_corr_vs_anchor': 0.5140093350003154}
{'candidate': 'realmlp_b1', 'fold': 0, 'anchor_qwk': np.float64(0.4684145364922969), 'blend_qwk': np.float64(0.467340301540203), 'gain_qwk': np.float64(-0.001074234952093911), 'anchor_b1_

...

RealMLP B1 done 196.9870936870575
{'candidate': 'catboost_b1', 'fold': 2, 'anchor_qwk': np.float64(0.478512778648449), 'blend_qwk': np.float64(0.4781073160893331), 'gain_qwk': np.float64(-0.0004054625591158878), 'anchor_b1_auc': np.float64(0.7785284915748898), 'specialist_b1_auc': np.float64(0.779440230539597), 'blend_b1_auc': np.float64(0.7794749882553194), 'alpha': 0.4, 'residual_corr_vs_anchor': 0.4239783234737771}
{'candidate': 'tabm_ple_b1', 'fold': 2, 'anchor_qwk': np.float64(0.478512778648449), 'blend_qwk': np.float64(0.4784202608732151), 'gain_qwk': np.float64(-9.251777523389126e-05), 'anchor_b1_auc': np.float64(0.7785284915748898), 'specialist_b1_auc': np.float64(0.7786265302047706), 'blend_b1_auc': np.float64(0.7802811237418648), 'alpha': 0.6000000000000001, 'residual_corr_vs_anchor': 0.36633668480447473}
{'candidate': 'realmlp_b1', 'fold': 2, 'anchor_qwk': np.float64(0.478512778648449), 'blend_qwk': np.float64(0.47961079391216), 'gain_qwk': np.float64(0.0010980152637110052),

Default metric period is 5 because AUC is/are not implemented for GPU


CatBoost B1 done 86.05641555786133
TabM epoch 01 loss=0.60209 cal_auc=0.776580 best=0.776580
TabM epoch 05 loss=0.55741 cal_auc=0.780287 best=0.780287
TabM epoch 10 loss=0.53492 cal_auc=0.780632 best=0.781158
TabM B1 done 41.7018609046936


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
`Trainer.fit` stopped: `max_epochs=128` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics 

RealMLP B1 done 202.36022400856018
{'candidate': 'catboost_b1', 'fold': 3, 'anchor_qwk': np.float64(0.4842608935593119), 'blend_qwk': np.float64(0.4799922673496927), 'gain_qwk': np.float64(-0.0042686262096192085), 'anchor_b1_auc': np.float64(0.782248389412165), 'specialist_b1_auc': np.float64(0.7827473790528532), 'blend_b1_auc': np.float64(0.7832607218617582), 'alpha': 1.2000000000000002, 'residual_corr_vs_anchor': 0.25074835825238706}
{'candidate': 'tabm_ple_b1', 'fold': 3, 'anchor_qwk': np.float64(0.4842608935593119), 'blend_qwk': np.float64(0.48419953710765773), 'gain_qwk': np.float64(-6.135645165417092e-05), 'anchor_b1_auc': np.float64(0.782248389412165), 'specialist_b1_auc': np.float64(0.7831707918777078), 'blend_b1_auc': np.float64(0.7838895101454237), 'alpha': 0.8, 'residual_corr_vs_anchor': 0.31483217138312064}
{'candidate': 'realmlp_b1', 'fold': 3, 'anchor_qwk': np.float64(0.4842608935593119), 'blend_qwk': np.float64(0.48287360966577153), 'gain_qwk': np.float64(-0.001387283893

## 12. Promotion gate: demand a qualitative jump

In [13]:
summary=[]
for cand,g in probe_df.groupby('candidate'):
    mean_auc=float(g.specialist_b1_auc.mean())
    mean_blend=float(g.blend_qwk.mean())
    mean_gain=float(g.gain_qwk.mean())
    wins=int((g.gain_qwk>0).sum())
    all_folds=(len(g)==len(CFG['PROBE_FOLDS']))
    # Promotion requires three-fold evidence, not one lucky probe fold.
    promote=bool(all_folds and wins>=CFG['PROMOTE_MIN_WINS'] and (
        mean_auc>=CFG['PROMOTE_MIN_B1_AUC'] or
        mean_blend>=CFG['PROMOTE_MIN_BLEND_QWK'] or
        mean_gain>=CFG['PROMOTE_MIN_QWK_GAIN']
    ))
    summary.append({
        'candidate':cand,'mean_specialist_b1_auc':mean_auc,'mean_blend_qwk':mean_blend,
        'mean_gain_qwk':mean_gain,'wins':wins,'mean_residual_corr':float(g.residual_corr_vs_anchor.mean()),
        'promote':promote,
        'breakthrough_auc_gap':CFG['TARGET_B1_AUC']-mean_auc,
    })
probe_summary=pd.DataFrame(summary).sort_values(['promote','mean_specialist_b1_auc','mean_blend_qwk'],ascending=[False,False,False])
probe_summary.to_csv(OUT/'v5_probe_summary.csv',index=False)
print(probe_summary.to_string(index=False))

promoted=probe_summary.loc[probe_summary.promote].candidate.tolist()
BEST_CANDIDATE=promoted[0] if promoted else None
print('\nOracle-derived B1 AUC breakthrough target:',CFG['TARGET_B1_AUC'])
print('BEST_CANDIDATE =',BEST_CANDIDATE)
if BEST_CANDIDATE is None:
    print('No candidate produced a qualitative multi-fold jump. Full 5-fold CV is intentionally skipped.')

      candidate  mean_specialist_b1_auc  mean_blend_qwk  mean_gain_qwk  wins  mean_residual_corr  promote  breakthrough_auc_gap
b1_cross_family                0.780069        0.476842      -0.000221     1            0.414357    False              0.112931
    tabm_ple_b1                0.779099        0.477649       0.000586     1            0.398393    False              0.113901
    catboost_b1                0.778708        0.475097      -0.001966     0            0.361308    False              0.114292
     realmlp_b1                0.775714        0.476608      -0.000455     1            0.504256    False              0.117286

Oracle-derived B1 AUC breakthrough target: 0.893
BEST_CANDIDATE = None
No candidate produced a qualitative multi-fold jump. Full 5-fold CV is intentionally skipped.


## 13. Full 5-fold run — only if a specialist survives

The full stage trains only the promoted candidate. For `b1_cross_family`, all available specialist families are trained and their calibration-only weights are reused on the untouched validation and test sets.

In [14]:
def train_specialist_predictions(candidate,fit_idx,cal_idx,val_idx,full,fold_id):
    cal={}; val={}; tst={}
    needed=['catboost_b1','tabm_ple_b1','realmlp_b1'] if candidate=='b1_cross_family' else [candidate]

    if 'catboost_b1' in needed:
        m=fit_cat_b1(fit_idx,cal_idx,full=full,seed=6100+fold_id)
        cal['catboost_b1']=m.predict_proba(CB_TRAIN.iloc[cal_idx])[:,1]
        val['catboost_b1']=m.predict_proba(CB_TRAIN.iloc[val_idx])[:,1]
        tst['catboost_b1']=m.predict_proba(CB_TEST)[:,1]
        del m; gc.collect()

    if 'tabm_ple_b1' in needed:
        m,prep,dev,_=fit_tabm_b1(fit_idx,cal_idx,full=full,seed=6200+fold_id)
        cal['tabm_ple_b1']=predict_tabm_dataframe(m,prep,dev,train.iloc[cal_idx])
        val['tabm_ple_b1']=predict_tabm_dataframe(m,prep,dev,train.iloc[val_idx])
        tst['tabm_ple_b1']=predict_tabm_dataframe(m,prep,dev,test)
        del m,prep; torch.cuda.empty_cache(); gc.collect()

    if 'realmlp_b1' in needed:
        if not REALMLP_AVAILABLE: raise RuntimeError('RealMLP promoted but unavailable')
        m,prep=fit_realmlp_b1(fit_idx,cal_idx,full=full,seed=6300+fold_id)
        cal['realmlp_b1']=predict_realmlp(m,prep,train.iloc[cal_idx])
        val['realmlp_b1']=predict_realmlp(m,prep,train.iloc[val_idx])
        tst['realmlp_b1']=predict_realmlp(m,prep,test)
        del m,prep; torch.cuda.empty_cache(); gc.collect()

    if candidate=='b1_cross_family':
        wf=search_b1_weights(y_b1[cal_idx],cal,6400+fold_id)
        return apply_b1_weights(cal,wf),apply_b1_weights(val,wf),apply_b1_weights(tst,wf),wf
    return cal[candidate],val[candidate],tst[candidate],None

FULL_CV_RAN=False
full_oof=np.full(len(train),np.nan,float)
full_oof_pred=np.zeros(len(train),int)
full_test_folds=[]
full_records=[]

if BEST_CANDIDATE is not None:
    FULL_CV_RAN=True
    for fold_id,(outer_tr,outer_va) in enumerate(folds):
        print('\n'+'='*100+f'\nFULL FOLD {fold_id} — {BEST_CANDIDATE}\n'+'='*100)
        fit_idx,cal_idx=inner_fit_cal_split(outer_tr,6000+fold_id)
        anchor=fit_cat_anchor(fit_idx,cal_idx,full=True,seed=6000+fold_id)
        bcal=anchor.predict(CB_TRAIN.iloc[cal_idx]); bval=anchor.predict(CB_TRAIN.iloc[outer_va]); btest=anchor.predict(CB_TEST)
        pcal,pval,ptest,wf=train_specialist_predictions(BEST_CANDIDATE,fit_idx,cal_idx,outer_va,True,fold_id)
        tune=tune_specialist_injection(y[cal_idx],bcal,pcal)
        sval=apply_specialist_injection(bval,pval,tune); stest=apply_specialist_injection(btest,ptest,tune)
        pred=apply_thresholds(sval,tune['thresholds'])
        full_oof[outer_va]=sval; full_oof_pred[outer_va]=pred; full_test_folds.append(stest)
        rec={
            'fold':fold_id,'qwk':qwk(y[outer_va],pred),'b1_auc':auc_b1(y[outer_va],sval),
            'specialist_b1_auc':roc_auc_score(y_b1[outer_va],pval),'alpha':tune['alpha'],
            'thresholds':tune['thresholds'].tolist(),
        }
        if wf is not None: rec['b1_weights']={n:float(w) for n,w in zip(wf['names'],wf['weights'])}
        full_records.append(rec); print(rec)
        del anchor; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    full_test=np.mean(full_test_folds,axis=0)
else:
    full_test=None

## 14. Honest full-CV decision and submission gate

In [15]:
SUBMISSION_ELIGIBLE=False
full_metrics={}
submission_path=None

if FULL_CV_RAN:
    assert np.isfinite(full_oof).all()
    # Fold-specific calibration predictions are already honest: each row's thresholds were fit only on its fold's calibration subset.
    cf_qwk=qwk(y,full_oof_pred)
    b1=auc_b1(y,full_oof)
    full_metrics={
        'candidate':BEST_CANDIDATE,
        'unique_id_crossfit_qwk':float(cf_qwk),
        'auc_gt1':float(b1),
        'target_top1':CFG['SUBMISSION_QWK_GATE'],
        'target_b1_auc':CFG['TARGET_B1_AUC'],
    }
    print('FULL honest metrics:',full_metrics)
    SUBMISSION_ELIGIBLE=bool(cf_qwk>=CFG['SUBMISSION_QWK_GATE'])
    print('SUBMISSION_ELIGIBLE =',SUBMISSION_ELIGIBLE)

    pd.DataFrame({'id':train[ID_COL],'y':y,'score':full_oof,'pred':full_oof_pred}).to_csv(OUT/'oof_v5.csv',index=False)
    pd.DataFrame(full_records).to_csv(OUT/'v5_full_fold_metrics.csv',index=False)
    pd.DataFrame({'id':test[ID_COL],'score':full_test}).to_csv(OUT/'test_scores_v5.csv',index=False)

    if SUBMISSION_ELIGIBLE:
        # Global thresholds are used only for the final test prediction after the honest gate is cleared.
        global_th=optimize_thresholds(y,full_oof)
        submission=pd.DataFrame({
            'id':test[ID_COL].to_numpy(),
            'life_satisfaction':apply_thresholds(full_test,global_th).astype(int),
        })
        assert len(submission)==CFG['EXPECTED_UNIQUE_TEST_IDS']
        assert submission.id.is_unique
        assert submission.life_satisfaction.isin([1,2,3,4]).all()
        submission_path=OUT/'submission_v5.csv'
        submission.to_csv(submission_path,index=False)
        print('SUBMISSION CREATED:',submission_path)
    else:
        print('NO SUBMISSION: honest CV did not reach 0.61541.')
else:
    print('No promoted boundary specialist. Full CV and submission remain locked.')

No promoted boundary specialist. Full CV and submission remain locked.


## 15. Diagnostics

In [16]:
if FULL_CV_RAN:
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    ax[0].hist(full_oof,bins=60,alpha=.8); ax[0].set_title('V5 OOF continuous score')
    ax[1].bar(['current','oracle-target'],[auc_b1(y,full_oof),CFG['TARGET_B1_AUC']]); ax[1].set_ylim(.7,.92); ax[1].set_title('Boundary-1 AUC')
    plt.tight_layout(); plt.show()

    cm=pd.crosstab(pd.Series(y,name='true'),pd.Series(full_oof_pred,name='pred'),normalize='index')
    print('Normalized confusion matrix')
    print(cm.round(4))
else:
    print('Diagnostics stop here because no candidate cleared the multi-probe breakthrough gate.')

Diagnostics stop here because no candidate cleared the multi-probe breakthrough gate.


## 16. Save summary safely

In [17]:
def json_default(obj):
    if isinstance(obj,np.generic): return obj.item()
    if isinstance(obj,np.ndarray): return obj.tolist()
    if isinstance(obj,Path): return str(obj)
    if isinstance(obj,set): return list(obj)
    if hasattr(obj,'item'):
        try:return obj.item()
        except Exception:pass
    if hasattr(obj,'tolist'):
        try:return obj.tolist()
        except Exception:pass
    return str(obj)

summary={
    'config':CFG,
    'realmlp_available':REALMLP_AVAILABLE,
    'probe_results':probe_df.to_dict(orient='records'),
    'probe_summary':probe_summary.to_dict(orient='records'),
    'best_candidate':BEST_CANDIDATE,
    'full_cv_ran':FULL_CV_RAN,
    'full_metrics':full_metrics,
    'submission_eligible':SUBMISSION_ELIGIBLE,
    'submission_path':submission_path,
}
with open(OUT/'v5_summary.json','w') as f:
    json.dump(summary,f,indent=2,default=json_default)
print('Artifacts saved to',OUT)

Artifacts saved to /kaggle/working/oqdq5_v5


## Reading the result

The notebook is intentionally conservative.

- If **no candidate is promoted**, do not spend a Kaggle submission. The model-side boundary hypothesis did not show the needed jump.
- If a candidate is promoted but full cross-fitted QWK is below **0.61541**, keep the artifacts for analysis but do not submit.
- If boundary AUC stays near **0.78–0.82**, the experiment strongly suggests that changing model family alone is not enough; the next investigation should move back to target/source-data generation rather than more hyperparameter tuning.
- Boundary AUC around **0.89** is the first region where the prior oracle experiment indicated that QWK 0.615 becomes mechanically plausible.